**Loading Secrets**

In [ ]:
# Loading the environments and libraries
%load_ext dotenv
%dotenv ../05_src/.secrets

**Importing Required Libraries**

In [31]:
from openai import OpenAI
import requests
import json
from langchain_community.document_loaders import PyPDFLoader
import os
import gradio as gr

**Service 1 - Real-time Weather Assistant**

In [32]:
# Initializing OpenAI FM
client_1 = OpenAI(base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1', 
                api_key='any value',
                default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')})

In [35]:
# Building the back-end of the service

# API function
def get_weather(city: str):
    url = f"http://api.weatherstack.com/current?access_key=e8439ac925148b7c736a4b831729364c&query={city}"
    response = requests.get(url)
    response.raise_for_status()
    return response.json()

# Define tool schema
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Get current weather for a city",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {
                        "type": "string",
                        "description": "City name, e.g. Toronto"
                    }
                },
                "required": ["city"]
            }
        }
    }
]

def run_model(user_prompt):
    messages = [
        {"role": "user", "content": user_prompt}
    ]

    response = client_1.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
        tools=tools
    )

    message = response.choices[0].message

    if message.tool_calls:
        tool_call = message.tool_calls[0]
        args = json.loads(tool_call.function.arguments)

        if tool_call.function.name == "get_weather":
            api_result = get_weather(args["city"])

            messages.append(message)
            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": json.dumps(api_result)
            })

            messages.append({
                "role": "system",
                "content": (
                    "Using the weather data provided, respond in exactly two sentences:\n"
                    "1. A short summary of the weather\n"
                    "2. A one-line clothing recommendation based on the weather"
                )
            })

            final_response = client_1.chat.completions.create(
                model="gpt-4o-mini",
                messages=messages
            )

            return final_response.choices[0].message.content

    return "I'm sorry, but I’m unable to help with that request right now."


demo = gr.Interface(
    fn=run_model,
    inputs=gr.Textbox(label="Enter your prompt"),
    outputs=gr.Textbox(label="Model response"),
    title="Weather Assistant",
    description="Ask about the weather in a city and get a clothing recommendation."
)

demo.launch()

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.
